In [21]:
import os
import glob
import h5py
import numpy as np
import pandas as pd
from scipy.optimize import curve_fit
from numpy.fft import rfft, rfftfreq


def estimate_baseline(y, n_samples=200):
    y0 = np.asarray(y, dtype=float)[:n_samples]
    return float(np.mean(y0)), float(np.std(y0))


def compute_energy_duration(waveform, threshold=0.9, n_baseline=200):
    y = np.asarray(waveform, dtype=float)

    baseline = np.mean(y[:n_baseline])
    y = y - baseline

    energy = y ** 2
    total_energy = float(np.sum(energy))

    if total_energy <= 0 or not np.isfinite(total_energy):
        return np.nan

    cumulative = np.cumsum(energy)
    target = threshold * total_energy

    idxs = np.where(cumulative >= target)[0]

    if len(idxs) == 0:
        return np.nan

    return int(idxs[0])


def compute_HWP(waveform, n_pre=200):

    y = np.asarray(waveform, dtype=float)

    baseline = float(np.mean(y[:n_pre]))
    y_rel = y - baseline

    peak_val = float(np.max(y_rel))

    if peak_val <= 0:
        return np.nan

    level25 = 0.25 * peak_val
    level75 = 0.75 * peak_val

    above_25 = np.where(y_rel >= level25)[0]
    above_75 = np.where(y_rel >= level75)[0]

    if len(above_25) == 0 or len(above_75) == 0:
        return np.nan

    left_idx = int(above_25[0])
    right_idx = int(above_75[-1])

    width = right_idx - left_idx

    if width < 0:
        return np.nan

    return float(width)


def compute_ND80(waveform, n_pre=200):

    y = np.asarray(waveform, dtype=float)

    baseline, _ = estimate_baseline(y, n_samples=n_pre)

    peak_idx = int(np.argmax(y))
    peak_val = float(y[peak_idx])

    amp = peak_val - baseline

    if amp <= 0:
        return np.nan

    level80 = baseline + 0.80 * amp

    above = np.where(y >= level80)[0]

    if len(above) == 0:
        return np.nan

    i80 = int(above[0])

    if i80 >= peak_idx:
        return 0.0

    seg = y[i80: peak_idx + 1]

    depth_vec = level80 - seg
    depth_vec[depth_vec < 0] = 0.0

    depth_abs = float(np.max(depth_vec))

    depth_norm = depth_abs / amp if amp > 0 else np.nan

    return depth_norm


def compute_PPR(waveform, n_plateau=300, n_baseline=200, eps=1e-12):

    y = np.asarray(waveform, dtype=float)

    baseline = float(np.mean(y[:n_baseline]))

    y0 = y - baseline

    peak_height = float(np.max(np.abs(y0)))

    if peak_height < eps:
        return np.nan

    plateau_height = float(np.abs(np.mean(y0[-n_plateau:])))

    return plateau_height / peak_height


def compute_frequency_spectrum(waveform, sample_spacing=1.0):

    wf = np.asarray(waveform, dtype=float)

    wf = wf - np.mean(wf[:200])

    N = len(wf)

    yf = rfft(wf)

    xf = rfftfreq(N, d=sample_spacing)

    amplitude = np.abs(yf) * 2.0 / N

    return xf, amplitude


def compute_spectral_centroid(waveform, sample_spacing=1.0):

    freqs, amp = compute_frequency_spectrum(waveform, sample_spacing)

    total_amp = np.sum(amp)

    if total_amp == 0:
        return 0.0

    centroid = np.sum(freqs * amp) / total_amp

    return float(centroid)


def exponential(t, a, tau1, b, tau2):

    return a * np.exp(-t / tau1) + b * np.exp(-t / tau2)


def pole_zero_correction(waveform, use_pz=True, min_tail_len=10):

    waveform = np.asarray(waveform, dtype=float)

    if not use_pz:
        return waveform.copy(), waveform.copy()

    peak_value = float(np.max(waveform))

    if (not np.isfinite(peak_value)) or peak_value <= 0:
        return waveform.copy(), waveform.copy()

    idx98 = np.where(waveform >= 0.98 * peak_value)[0]

    if len(idx98) == 0:
        return waveform.copy(), waveform.copy()

    t98 = int(idx98[0])

    tail_values = waveform[t98:]

    tail_time = np.arange(tail_values.size, dtype=float)

    if tail_values.size < min_tail_len:
        return waveform.copy(), tail_values.copy()

    p0 = [tail_values[0], 300.0, tail_values[0] * 0.1, 1500.0]

    try:

        params, _ = curve_fit(
            exponential,
            tail_time,
            tail_values,
            p0=p0,
            maxfev=20000
        )

    except:

        return waveform.copy(), tail_values.copy()

    f_decay = exponential(tail_time, *params)

    end = min(t98 + 5, len(waveform))

    f_t0 = float(np.mean(waveform[t98:end]))

    eps = 1e-12

    f_pz = f_t0 / (f_decay + eps)

    waveform_tail_corrected = tail_values * f_pz

    waveform_pz = waveform.copy()

    waveform_pz[t98:] = waveform_tail_corrected

    return waveform_pz, waveform_tail_corrected


def compute_LQ80(waveform):

    waveform_pz, _ = pole_zero_correction(waveform, use_pz=True)

    y = np.asarray(waveform, dtype=float)
    yc = np.asarray(waveform_pz, dtype=float)

    baseline, _ = estimate_baseline(y)

    peak_val = float(np.max(y))

    target = baseline + 0.80 * (peak_val - baseline)

    idx = np.where(y >= target)[0]

    if len(idx) == 0:
        return np.nan

    i80 = int(idx[0])

    t = np.arange(len(y), dtype=float)

    area_raw = float(np.trapz(y[i80:], t[i80:]))

    area_corr = float(np.trapz(yc[i80:], t[i80:]))

    return area_raw - area_corr

In [22]:
def process_hdf5_file(h5_path, out_dir):

    print(f"Processing {h5_path}...")

    basename = os.path.basename(h5_path)

    file_idx = basename.split("_")[2].split(".")[0] 

    with h5py.File(h5_path, "r") as f:
        waveforms = f["raw_waveform"][:]
        ids = f["id"][:]

    n_events = waveforms.shape[0]

    print(f"Found {n_events} waveforms")

    ED_list = []
    HWP_list = []
    ND80_list = []
    PPR_list = []
    SCA_list = []
    LQ80_list = []

    for i in range(n_events):

        wf = waveforms[i]

        ED_list.append(compute_energy_duration(wf))
        HWP_list.append(compute_HWP(wf))
        ND80_list.append(compute_ND80(wf))
        PPR_list.append(compute_PPR(wf))
        SCA_list.append(compute_spectral_centroid(wf))
        LQ80_list.append(compute_LQ80(wf))

        if (i + 1) % 5000 == 0:
            print(f"Processed {i+1}/{n_events}")

    df = pd.DataFrame({
        "id":  [f"{ids[i]}_npml_{file_idx}" for i in range(len(ids))],
        "file": basename,
        "ED": ED_list,
        "HWP": HWP_list,
        "ND80": ND80_list,
        "PPR": PPR_list,
        "SCA": SCA_list,
        "LQ80": LQ80_list
    })

    os.makedirs(out_dir, exist_ok=True)

    out_name = os.path.splitext(basename)[0] + "_eunice_params.csv"
    out_path = os.path.join(out_dir, out_name)

    df.to_csv(out_path, index=False)

    print("Saved:", out_path)

In [23]:
def main():

    DATA_DIR = os.path.abspath("../../data/old")

    NPML_PATTERN = os.path.join(DATA_DIR, "MJD_NPML*.hdf5")

    OUT_DIR = os.path.join(DATA_DIR, "params_npml_eunice")

    npml_files = sorted(glob.glob(NPML_PATTERN))

    print("NPML files:", npml_files)

    for path in npml_files:
        process_hdf5_file(path, OUT_DIR)


if __name__ == "__main__":
    main()

NPML files: ['c:\\Users\\YooNi\\OneDrive\\Desktop\\Majorana-Neutrino-Hunt\\data\\old\\MJD_NPML_0.hdf5', 'c:\\Users\\YooNi\\OneDrive\\Desktop\\Majorana-Neutrino-Hunt\\data\\old\\MJD_NPML_1.hdf5', 'c:\\Users\\YooNi\\OneDrive\\Desktop\\Majorana-Neutrino-Hunt\\data\\old\\MJD_NPML_2.hdf5']
Processing c:\Users\YooNi\OneDrive\Desktop\Majorana-Neutrino-Hunt\data\old\MJD_NPML_0.hdf5...
Found 65000 waveforms


C:\Users\YooNi\AppData\Local\Temp\ipykernel_16188\3820234494.py:245: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  area_raw = float(np.trapz(y[i80:], t[i80:]))
C:\Users\YooNi\AppData\Local\Temp\ipykernel_16188\3820234494.py:247: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  area_corr = float(np.trapz(yc[i80:], t[i80:]))
C:\Users\YooNi\AppData\Local\Temp\ipykernel_16188\3820234494.py:159: RuntimeWarning: overflow encountered in exp
  return a * np.exp(-t / tau1) + b * np.exp(-t / tau2)
C:\Users\YooNi\AppData\Local\Temp\ipykernel_16188\3820234494.py:159: RuntimeWarning: overflow encountered in multiply
  return a * np.exp(-t / tau1) + b * np.exp(-t / tau2)


Processed 5000/65000
Processed 10000/65000
Processed 15000/65000
Processed 20000/65000
Processed 25000/65000
Processed 30000/65000
Processed 35000/65000
Processed 40000/65000
Processed 45000/65000
Processed 50000/65000
Processed 55000/65000
Processed 60000/65000
Processed 65000/65000
Saved: c:\Users\YooNi\OneDrive\Desktop\Majorana-Neutrino-Hunt\data\old\params_npml_eunice\MJD_NPML_0_eunice_params.csv
Processing c:\Users\YooNi\OneDrive\Desktop\Majorana-Neutrino-Hunt\data\old\MJD_NPML_1.hdf5...
Found 65000 waveforms
Processed 5000/65000
Processed 10000/65000
Processed 15000/65000
Processed 20000/65000
Processed 25000/65000
Processed 30000/65000
Processed 35000/65000
Processed 40000/65000
Processed 45000/65000


C:\Users\YooNi\AppData\Local\Temp\ipykernel_16188\3820234494.py:192: OptimizeWarning: Covariance of the parameters could not be estimated
  params, _ = curve_fit(


Processed 50000/65000
Processed 55000/65000
Processed 60000/65000
Processed 65000/65000
Saved: c:\Users\YooNi\OneDrive\Desktop\Majorana-Neutrino-Hunt\data\old\params_npml_eunice\MJD_NPML_1_eunice_params.csv
Processing c:\Users\YooNi\OneDrive\Desktop\Majorana-Neutrino-Hunt\data\old\MJD_NPML_2.hdf5...
Found 29697 waveforms
Processed 5000/29697
Processed 10000/29697
Processed 15000/29697
Processed 20000/29697
Processed 25000/29697
Saved: c:\Users\YooNi\OneDrive\Desktop\Majorana-Neutrino-Hunt\data\old\params_npml_eunice\MJD_NPML_2_eunice_params.csv


In [24]:
file_list = [
"../../data/old/params_npml_eunice/MJD_NPML_0_eunice_params.csv",
"../../data/old/params_npml_eunice/MJD_NPML_1_eunice_params.csv",
"../../data/old/params_npml_eunice/MJD_NPML_2_eunice_params.csv"
]

df_list = []

for file in file_list:
    df = pd.read_csv(file)
    df_list.append(df)

combined_df = pd.concat(df_list, ignore_index=True)

combined_df.to_csv(
    "eunice_combined_npml.csv.gz",
    index=False,
    compression="gzip"
)

print("Saved eunice_combined_npml.csv.gz")

Saved eunice_combined_npml.csv.gz


In [25]:
print("Combined shape:", combined_df.shape)

Combined shape: (159697, 8)


In [26]:
combined_df

,id,file,ED,HWP,ND80,PPR,SCA,LQ80
0,3033789_npml_0,MJD_NPML_0.hdf5,3405,2090.0,0.0,0.694995,0.034962,-3.234871e+05
1,3033790_npml_0,MJD_NPML_0.hdf5,3406,2165.0,0.0,0.702777,0.035541,-2.097532e+05
2,3033791_npml_0,MJD_NPML_0.hdf5,3408,2129.0,0.0,0.699074,0.035238,-2.425070e+05
3,3033792_npml_0,MJD_NPML_0.hdf5,3411,2125.0,0.0,0.701728,0.035928,-2.223822e+05
4,3033793_npml_0,MJD_NPML_0.hdf5,3405,2002.0,0.0,0.684008,0.035634,-2.887550e+05
...,...,...,...,...,...,...,...,...
159692,3193481_npml_2,MJD_NPML_2.hdf5,3408,2046.0,0.0,0.693882,0.034458,-8.314132e+05
159693,3193482_npml_2,MJD_NPML_2.hdf5,3411,2060.0,0.0,0.696157,0.034467,-1.789650e+06
159694,3193483_npml_2,MJD_NPML_2.hdf5,3409,2082.0,0.0,0.697220,0.034344,-1.315494e+06
159695,3193484_npml_2,MJD_NPML_2.hdf5,3405,2062.0,0.0,0.693403,0.034225,-2.350121e+06


In [27]:
print(combined_df.head())

               id             file    ED     HWP  ND80       PPR       SCA  \
0  3033789_npml_0  MJD_NPML_0.hdf5  3405  2090.0   0.0  0.694995  0.034962   
1  3033790_npml_0  MJD_NPML_0.hdf5  3406  2165.0   0.0  0.702777  0.035541   
2  3033791_npml_0  MJD_NPML_0.hdf5  3408  2129.0   0.0  0.699074  0.035238   
3  3033792_npml_0  MJD_NPML_0.hdf5  3411  2125.0   0.0  0.701728  0.035928   
4  3033793_npml_0  MJD_NPML_0.hdf5  3405  2002.0   0.0  0.684008  0.035634   

            LQ80  
0 -323487.123581  
1 -209753.179106  
2 -242506.954320  
3 -222382.185659  
4 -288754.972322  
